## This notebook will test using a tensor DNN to try to get similar performance to the paper

Will attempt to get a testing implementation working to try optimising parameters.

In [18]:
import time
from collections.abc import Sized, Iterable

import optuna
from optuna import Trial
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

#### Configuration

Notes among variables

* Batch normalisation reduces need for dropout highlighted in original paper
* Dropout still appears to help at lower percentage
* Adding depth to DNN much better than adding width in manual testing
* Dropout of first layer is a disaster and I don't know why I thought it would be good

In [2]:
#
csv_path = "./HIGGS.csv"

# --- feature set -----------------------------------------------------
# 28 columns total. Columns 1-21 are the "low-level" kinematic features
# columns 22-28 are the 7 "high-level" derived features (m_jj ... m_wwbb)
feature_set = "low"               # "all" | "low"

# --- architecture ----------------------------------------------------
# Paper's best: "five-layer" net = 4 hidden x 300 + output
# hidden_layers = [300, 300, 300, 300, 300, 300, 300]
# hidden_layers = [600, 600, 600, 600]
# GELU appears best!
#act_layer = nn.GELU # nn.ReLU, nn.SiLU nn.Tanh, or possibly nn.GELU

# with no batch normalisation, more dropout is better - 50% better than 20%
# batchnorm enabled with no dropout is significantly better than 50% dropout without batch normalisation
# batch normalisation plus 50% dropout is a slight improvement, but 20% is a bigger improvement
# batch normalisation mitigates the need for dropout and makes a smaller percentage perform better
#batchnorm = True                  # stabiliser see paper https://arxiv.org/abs/1502.03167 - nn.BatchNorm1d

#dropout = 0.2                     # paper applied 50% dropout to the TOP hidden layer
#initial_dropout = 0.0


# --- optimisation ----------------------------------------------------
epochs = 100                       # 100 assuming we will reach patience limit prior
batch_size = 8192                 # big batches keep the GPU busy; paper used 100 (2011 GPU)
# max_lr = 0.003                     # OneCycle peak LR for AdamW
max_lr = 0.005                     # OneCycle peak LR for AdamW
weight_decay = 0.00001               # matches the paper's L2 coefficient
label_smoothing = 0.0             # try 0.01-0.05 if the net overfits

# --- engineering -----------------------------------------------------
amp = True                        # bfloat16 autocast
compile = True                    # torch.compile fuses kernels; big speedup, falls back if it fails
keep_data_on_gpu = True
early_stop_patience = 6           # stop if val AUC hasn't improved for this many epochs
seed = 1337

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Read and split the data

In [3]:

data = pd.read_csv(csv_path, header=None, dtype=np.float32)

arr = data.to_numpy()


In [10]:
y = arr[:, 0].astype(np.int8)      # first column = label
X = arr[:, 1:].astype(np.float32)  # remaining 28 columns = features

# Select the feature subset (mirrors the paper's three experiments).
if feature_set == "low":
    X = X[:, :21]                  # 21 low-level kinematic features

X_train, X_val1, y_train, y_val1 = train_test_split(X, y, test_size=0.0909090909, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_val1, y_val1, test_size=0.5, random_state=42)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)


# X_train_small, X_val1_small, y_train_small, y_val1_small = train_test_split(X_small, y_small, test_size=0.0909090909, random_state=42)
# X_val_small, X_test_small, y_val_small, y_test_small = train_test_split(X_val1_small, y_val1_small, test_size=0.5, random_state=42)

(10000000, 21)
(500000, 21)
(500000, 21)
(10000000,)
(500000,)
(500000,)


#### Model class

Creates the DNN

In [15]:

class HiggsMLP(nn.Module):

    def __init__(self,
                 in_dim: int,
                 hidden_layers: list[int],
                 last_layer_dropout: float,
                 act_layer
             ):
        super().__init__()

        layers = []
        prev = in_dim
        n_hidden = len(hidden_layers)
        for i, width in enumerate(hidden_layers):
            layers.append(nn.Linear(prev, width))
            # batchnorm consistently better
            layers.append(nn.BatchNorm1d(width))
            layers.append(act_layer())
            is_last_hidden = (i == n_hidden - 1)
            if last_layer_dropout > 0 and is_last_hidden:
                layers.append(nn.Dropout(last_layer_dropout))
            prev = width

        layers.append(nn.Linear(prev, 1))    # single logit for binary classification
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)        # shape [B] to match the label vector

#### Trainer

Method to use the above class to train a model. Using a static method so we can pass everything in, and don't accidentally use a variable from the global scope.

In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)
# TF32 matmuls: free accuracy-for-speed trade on NVIDIA GPUs for fp32 ops.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [16]:

# No-grad - use for inference to reduce memory
@torch.no_grad()
def compute_auc(model, X_gpu, y_np, batch: int = 65536):
    model.eval()
    probs = []
    for i in range(0, len(X_gpu), batch):
        logits = model(X_gpu[i:i + batch])
        probs.append(torch.sigmoid(logits).float().cpu().numpy())
    return roc_auc_score(y_np, np.concatenate(probs))

def build_hidden(trial) -> list[int]:
    depth = trial.suggest_int("depth", 3, 6)
    base = trial.suggest_categorical("base_width", [256, 384, 512, 640, 768, 1024])
    shape = trial.suggest_categorical("shape", ["uniform", "pyramid"])
    if shape == "uniform":
        return [base] * depth
    # pyramid: halve each layer, floor at 64
    return [max(64, base // (2 ** i)) for i in range(depth)]

class TrainingHelper:

    @staticmethod
    def train(
            trial: Trial,
            hidden_layers: list[int],
            X_train, # : arr[arr[float]], would really love to give these data types but honestly it's unclear what they should be
            y_train, # : arr[arr[float]],
            X_val, # : ndarray[torch.float32],
            y_val,
            act_func,
            epochs: int,
            max_lr: float,
            weight_decay: float,
            last_layer_dropout: float,
            batch_size: int,
            device,
    ):

        to_gpu = True
        Xtr_t = torch.tensor(X_train, device=device if to_gpu else "cpu")
        ytr_t = torch.tensor(y_train, dtype=torch.float32, device=device if to_gpu else "cpu")
        Xva_t = torch.tensor(X_val, device=device)

        # ---- model / loss / optimiser --------------------------------------
        model = HiggsMLP(
            in_dim=X_train.shape[1],
            hidden_layers=hidden_layers,
            last_layer_dropout=last_layer_dropout,
            act_layer=act_func
        ).to(device)

        if compile:
            try:
                model = torch.compile(model)
                print("torch.compile: enabled")
            except Exception as e:
                print(f"torch.compile failed ({e}); continuing uncompiled")

        # BCEWithLogitsLoss = sigmoid + binary cross-entropy, fused & numerically stable.
        loss_fn = nn.BCEWithLogitsLoss()
        # AdamW is the modern default (decoupled weight decay)
        opt = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)
        # Basic, similar to original paper
        # opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=1e-5)

        n_train = len(Xtr_t)
        steps_per_epoch = (n_train + batch_size - 1) // batch_size

        # schedule for MLPs; a modern replacement for the paper's slow manual decay.
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=max_lr,
            epochs=epochs, steps_per_epoch=steps_per_epoch,
            pct_start=0.1  # was having an issue where patience would run out before lr would decay - want to keep training for up to 100 but likely will never go that long
        )

        # bfloat16 autocast: half-precision math on the GPU, ~2x throughput. bf16
        # (unlike fp16) has enough dynamic range that NO GradScaler is needed.
        amp_ctx = (torch.autocast("cuda", dtype=torch.bfloat16)
                   if amp and DEVICE.type == "cuda"
                   else torch.autocast("cuda", enabled=False))

        best_auc, best_state, epochs_no_improve = 0.0, None, 0
        for epoch in range(epochs):
            model.train()
            t0 = time.perf_counter()
            perm = torch.randperm(n_train, device=Xtr_t.device)  # fresh shuffle each epoch
            running = 0.0
            for i in range(0, n_train, batch_size):
                idx = perm[i:i + batch_size]
                xb = Xtr_t[idx]
                yb = ytr_t[idx]
                if not to_gpu:                       # only needed if data is on CPU
                    xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)

                opt.zero_grad(set_to_none=True)      # set_to_none is slightly faster than zeroing
                with amp_ctx:
                    logits = model(xb)
                    loss = loss_fn(logits, yb)
                loss.backward()
                opt.step()
                sched.step()                         # OneCycle steps PER BATCH, not per epoch
                running += loss.item() * len(idx)

            val_auc = compute_auc(model, Xva_t, y_val)
            dt = time.perf_counter() - t0

            # Early stopping: keep the best-on-validation weights, stop if stalled.
            if val_auc > best_auc + 0.0001:
                best_auc = val_auc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            trial.report(val_auc, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

            #     epochs_no_improve = 0
            # else:
            #     epochs_no_improve += 1
            #     if epochs_no_improve >= early_stop_patience:
            #         print(f"early stopping (no val improvement for {early_stop_patience} epochs)")
            #         break

        return best_auc, best_state


activation_types = {"relu": nn.ReLU, "tanh": nn.Tanh, "gelu": nn.GELU}
BATCHES = [8192, 16384, 32768]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    hidden = build_hidden(trial)
    act = trial.suggest_categorical("act", ["relu", "gelu"])
    dropout = trial.suggest_float("dropout", 0.0, 0.4)
    lr = trial.suggest_float("max_lr", 1e-3, 4e-3, log=True)
    wd = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    batch = trial.suggest_categorical("batch", BATCHES)

    torch.manual_seed(0)
    np.random.seed(0)

    best_auc, best_model = TrainingHelper.train(
        trial,
        hidden,
        X_train,
        y_train,
        X_val,
        y_val,
        activation_types[act],
        epochs,
        lr,
        wd,
        dropout,
        batch,
        device=DEVICE
    )

    print(f"Layers: {len(hidden)} Size: {hidden[0]} Block {hidden[1]==hidden[0]} AUC {best_auc}")

    return best_auc        # Optuna maximizes the best validation AUC of this config


### Optimisation

Use optuna to run hyperparam trials

In [ ]:
    study = optuna.create_study(
        direction="maximize",
        study_name=f"higgs_{FEATURE_SET}",
        storage=f"sqlite:///higgs_optuna.db",   # persistent -> resumable across restarts
        load_if_exists=True,
        sampler=optuna.samplers.TPESampler(multivariate=True, seed=0),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=5),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
